# P2 D-256 StyleGAN2 — Colab Training

**Target:** FFHQ 256×256 from scratch, 14M images, FID 5~15.
**Hardware:** Colab Pro+ A100 (952 compute units available → ~79h A100).
**Resume strategy:** ckpt every 200k images, Drive backup, 24h session reconnect.

Before running:
1. Runtime → Change runtime type → **A100 GPU**
2. Upload `train_50k_256.zip` to Drive at `MyDrive/p2-data/train_50k_256.zip` (1.65GB)
3. Optional: `valid_10k_256.zip` to same folder for FID self-measurement

In [ ]:
# 1. GPU check — must be A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone repo (or pull latest if already cloned)
import os
if not os.path.exists('/content/osai'):
    !git clone https://github.com/geniemo/osai.git /content/osai
%cd /content/osai
!git checkout improve
!git pull

In [ ]:
# 4. Install dependencies
!pip install -q pyyaml wandb pytorch-fid onnx onnxruntime scipy

In [ ]:
# 5. Copy training zip from Drive to Colab local disk (faster I/O)
!mkdir -p /content/osai/p2/data
!cp /content/drive/MyDrive/p2-data/train_50k_256.zip /content/osai/p2/data/
!ls -la /content/osai/p2/data/

In [ ]:
# 6. WandB login
import wandb
wandb.login()

In [ ]:
# 7. Make run dir backup point on Drive
!mkdir -p /content/drive/MyDrive/p2-runs/d256_main

## Launch — first run

If this is the first session: run the next cell.
If you're resuming after a disconnect: skip to the **Resume** section below.

In [ ]:
%cd /content/osai
!PYTHONPATH=. python p2/train.py --config p2/configs/d256.yaml 2>&1 | tee -a /content/drive/MyDrive/p2-runs/d256_main/train.log

## Backup checkpoints to Drive (run periodically in a separate Colab tab)

Open a second Colab tab on the same A100 to mirror ckpts to Drive every few minutes.

In [ ]:
# Periodic backup — run in a separate cell/tab
import time, subprocess
while True:
    subprocess.run(['rsync', '-av', '--update',
                    '/content/osai/p2/runs/d256_main/',
                    '/content/drive/MyDrive/p2-runs/d256_main/'])
    print('[backup] sync done, sleeping 10min')
    time.sleep(600)

## Resume — after disconnect

1. Run cells 1-7 again (mount Drive, clone, install, copy zip, wandb).
2. Copy back the latest ckpt from Drive:
```
!mkdir -p /content/osai/p2/runs/d256_main
!cp /content/drive/MyDrive/p2-runs/d256_main/ckpt_*.pt /content/osai/p2/runs/d256_main/
```
3. Find latest ckpt and resume:

In [ ]:
# Resume launch
import glob
ckpts = sorted(glob.glob('/content/osai/p2/runs/d256_main/ckpt_*.pt'))
latest = ckpts[-1] if ckpts else None
print('Latest ckpt:', latest)
if latest:
    %cd /content/osai
    !PYTHONPATH=. python p2/train.py --config p2/configs/d256.yaml --resume {latest} 2>&1 | tee -a /content/drive/MyDrive/p2-runs/d256_main/train.log

## Self-measure FID (run between sessions)

Requires `valid_10k_256.zip` extracted as a directory. Real-stats cached once.

In [ ]:
# One-time: extract valid + cache FID real stats
!cp /content/drive/MyDrive/p2-data/valid_10k_256.zip /content/osai/p2/data/
!mkdir -p /content/osai/p2/data/valid_10k_256_dir
!cd /content/osai/p2/data/valid_10k_256_dir && unzip -q -o ../valid_10k_256.zip
!cd /content/osai && python -m pytorch_fid p2/data/valid_10k_256_dir --save-stats p2/checkpoints/fid_stats_256.npz

In [ ]:
# Measure FID against latest ckpt (or specified ckpt)
import glob
ckpts = sorted(glob.glob('/content/osai/p2/runs/d256_main/ckpt_*.pt'))
latest = ckpts[-1] if ckpts else None
print('Evaluating:', latest)
if latest:
    !cd /content/osai && PYTHONPATH=. python p2/eval_fid.py --ckpt {latest} --stats p2/checkpoints/fid_stats_256.npz --n 8000 --batch 32

## ONNX export — for leaderboard submission

In [ ]:
# Export final.pt (or specified ckpt) to ONNX
!cd /content/osai && PYTHONPATH=. python p2/export_onnx.py \
    --ckpt p2/runs/d256_main/final.pt \
    --out p2/checkpoints/model.onnx
# Copy to Drive for download
!cp /content/osai/p2/checkpoints/model.onnx /content/drive/MyDrive/p2-runs/d256_main/